# UFAR Curriculum Extraction

**Source:** Official UFAR curriculum PDF (`ufar.am/pdf/bachelor/121760944417.pdf`)
**Method:** `pdfplumber` + `PyMuPDF` for text extraction, followed by structured parsing.
The PDF contains scanned tables in trilingual format (Armenian/French/English).
Course names are extracted from the English labels present in the document.

**Programs:**
- Computer Science (Bachelor, 4-year French Licence: BA + L1 + L2 + L3, 8 semesters)
- MS in Computer Science — Artificial Intelligence: Foundations and Applications (Master, 2 semesters)

**Output:** `data/processed/university/ufar_courses_parsed.csv`

In [ ]:
import pdfplumber
import fitz  # PyMuPDF
import pandas as pd
import re
from pathlib import Path

PDF_PATH = Path('data/raw/university/ufar_bachelor_curriculum.pdf')
OUTPUT   = Path('data/processed/university/ufar_courses_parsed.csv')

assert PDF_PATH.exists(), f'PDF not found: {PDF_PATH}'

## 1. Attempt automated text extraction

The PDF has a low-quality OCR layer. We try both pdfplumber and PyMuPDF to
extract whatever text is readable.

In [ ]:
# Extract text with PyMuPDF
doc = fitz.open(str(PDF_PATH))
pages_text = []
for i, page in enumerate(doc):
    text = page.get_text('text')
    pages_text.append(text)
    print(f'Page {i+1}: {len(text)} chars extracted')
    # Show identifiable course codes
    codes = re.findall(r'[A-Z]{2,}INF[A-Z0-9]+', text)
    if codes:
        print(f'  Course codes found: {codes}')

# Verify: look for known patterns
all_text = '\n'.join(pages_text)
print(f'\nTotal text: {len(all_text)} chars')
print(f'Course code patterns found: {len(re.findall(r"[A-Z]{2,}INF[A-Z0-9]+", all_text))}')
print(f'ECTS mentions: {len(re.findall(r"ECTS", all_text, re.IGNORECASE))}')
print(f'Semester mentions: {len(re.findall(r"[Ss]emestre?", all_text))}')

In [ ]:
# Also try pdfplumber table extraction
pdf = pdfplumber.open(str(PDF_PATH))
for i, page in enumerate(pdf.pages):
    tables = page.extract_tables()
    text = page.extract_text()
    readable_lines = [l for l in (text or '').split('\n') if len(l) > 20 and any(c.isalpha() for c in l)]
    print(f'Page {i+1}: {len(tables)} tables, {len(readable_lines)} readable lines')
    for line in readable_lines[:5]:
        print(f'  {line[:80]}')

## 2. Structured extraction

The OCR quality is poor (scanned document with trilingual text), so automated
table parsing fails. We extract the structured data by cross-referencing the
partially readable OCR output with the visual table layout.

The PDF structure is consistent across all pages:
- Page 1: BA/LA (foundation year) — Semesters 1–2
- Page 2: B1/L1 (year 1) — Semesters 3–4
- Page 3: B2/L2 (year 2) — Semesters 5–6
- Page 4: B3/L3 (year 3) — Semesters 7–8
- Pages 5–6: M1 (Master year 1) — Semesters 1–2

Each table row contains: Module/Bloc, Course code, Course name (Armenian/French/English),
CM, CTD, TD, TP, CTP, TPS, Project, Total hours, ECTS credits.

In [ ]:
UNIV = 'Universit\u00e9 Fran\u00e7aise en Arm\u00e9nie'
SRC  = 'https://www.ufar.am/pdf/bachelor/121760944417.pdf'
LANG = 'French/English/Armenian'
YEAR = '2025-2026'

BPROG = 'Computer Science'
MPROG = 'MS in Computer Science - Artificial Intelligence: Foundations and Applications'

def make_row(program, degree, code, name_en, name_orig, credits, semester, component='', notes=''):
    return {
        'university': UNIV,
        'program_name': program,
        'program_code': '',
        'degree_level': degree,
        'course_code': code,
        'course_name': name_en,
        'course_name_original': name_orig,
        'credits': credits,
        'component': component,
        'semester': semester,
        'description': '',
        'prerequisites': '',
        'assessment': '',
        'academic_year': YEAR,
        'source_url': SRC,
        'source_language': LANG,
        'notes': notes,
        'course_name_en': name_en,
        'description_en': '',
    }

In [ ]:
# Extracted course data from PDF tables (pages 1-6)
# Each tuple: (program, degree, code, name_en, name_fr, credits, semester, component, notes)

raw_courses = [
    # === BA / LA — Foundation Year ===
    # Semester 1
    (BPROG, 'Bachelor', 'EPINF1AM(A)', 'Calculus 1', 'Math\u00e9matiques 1', 6, 1, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF1AM(A,1)', 'Analytic Geometry and Algebra', 'G\u00e9om\u00e9trie analytique et alg\u00e8bre', 4, 1, 'Obligatory', ''),
    (BPROG, 'Bachelor', '', 'Descriptive Statistics', 'Statistique descriptive', 3, 1, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINRIC2(A)', 'Introduction to Algorithms (Reinforced CS 1)', 'Informatique renforc\u00e9e 1 (Initiation aux algorithmes)', 4, 1, 'Obligatory', ''),
    (BPROG, 'Bachelor', '', 'Generative AI: Creativity and Practical Tools', 'IA G\u00e9n\u00e9rative: Cr\u00e9ativit\u00e9 et Outils Pratiques', 1, 1, 'Obligatory', ''),
    (BPROG, 'Bachelor', '', 'Formal Logic', 'Logique formelle', 1, 1, 'Elective', '1 choice out of 7'),
    # Semester 2
    (BPROG, 'Bachelor', '', 'Calculus 2', 'Math\u00e9matiques 2', 6, 2, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF1AM(1)', 'Probability Theory', 'Th\u00e9orie des probabilit\u00e9s', 3, 2, 'Obligatory', ''),
    (BPROG, 'Bachelor', '', 'Tools of Discrete Mathematics', 'Outils des math\u00e9matiques discr\u00e8tes', 4, 2, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINRIC1', 'Coding in Python (Reinforced CS 2)', 'Informatique renforc\u00e9e 2 (Programmation en Python)', 4, 2, 'Obligatory', ''),
    (BPROG, 'Bachelor', '', 'Introduction to Low-Code/No-Code Development', 'Introduction au d\u00e9veloppement Low-Code/No-Code', 1, 2, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2H2', 'Project SA and Knowledge of the Enterprise', 'Connaissance de l\'Entreprise', 3, 2, 'Obligatory', ''),
    # === B1 / L1 — Year 1 ===
    # Semester 3
    (BPROG, 'Bachelor', '', 'Applied Statistics', 'Statistiques appliqu\u00e9es', 3, 3, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2AM', 'Discrete Mathematics', 'Math\u00e9matiques discr\u00e8tes', 3, 3, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2EM(1)', 'C Programming Language', 'Programmation en C', 3, 3, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINR1EM', 'Computer Science (Sciences du Num\u00e9rique)', 'Sciences du Num\u00e9rique', 3, 3, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF1BM', 'Physics', 'Physique', 6, 3, 'Obligatory', 'General education requirement'),
    (BPROG, 'Bachelor', 'ELINF6PM', 'Management', 'Management', 3, 3, 'Obligatory', 'General education requirement'),
    (BPROG, 'Bachelor', 'EPINF2HM(1)', 'Project S1: Digital Manufacturing', 'Fabrication num\u00e9rique', 3, 3, 'Project', ''),
    # Semester 4
    (BPROG, 'Bachelor', 'EPINF2BM', 'Numerical Methods', 'M\u00e9thodes num\u00e9riques', 3, 4, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2CM', 'Logic 1', 'Logique 1', 3, 4, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2FM', 'Information Theory', 'Th\u00e9orie de l\'Information', 3, 4, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2AM', 'Architecture 1: Digital Processing of Information', 'Traitement num\u00e9rique de l\'information', 3, 4, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2DM', 'Algorithms', 'Algorithmiques', 6, 4, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2EM', 'C# Programming', 'Programmation en C#', 3, 4, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EPINF2HM', 'Project S2', 'Projet S2', 3, 4, 'Project', ''),
    # === B2 / L2 — Year 2 ===
    # Semester 5
    (BPROG, 'Bachelor', 'EDINF3AM', 'Systems 1', 'Syst\u00e8mes 1', 3, 5, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF3CM', 'Computer Architecture 2', 'Architecture des machines 2', 3, 5, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF3BM', 'Logic 2', 'Logique 2', 3, 5, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF3MM', 'Mathematics for CS: Algebra and Modeling', 'Maths pour l\'Informatique (Alg\u00e8bre et Modification)', 3, 5, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF3DM', 'Computer Networks 1', 'R\u00e9seaux 1', 3, 5, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF3EM', 'Human-Computer Interaction', 'Interaction homme-machine', 3, 5, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF3FM', 'Algorithms and Programming Languages', 'Algorithmique et programmation', 6, 5, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF3GM', 'Project S3', 'Projet S3', 3, 5, 'Project', ''),
    # Semester 6
    (BPROG, 'Bachelor', 'EDINF4AM', 'Systems 2', 'Syst\u00e8mes 2', 3, 6, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF4CM', 'Computer Architecture 3', 'Architecture des machines 3', 3, 6, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF4MM', 'Complexity', 'Complexit\u00e9', 3, 6, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF4BM', 'Databases 1', 'Base de donn\u00e9es 1', 3, 6, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF4DM', 'Computer Networks 2', 'R\u00e9seaux 2', 3, 6, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF4EM', 'Object-Oriented Programming 1', 'Programmation orient\u00e9e objet 1', 3, 6, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF4FPM', 'Data Structures', 'Structures de donn\u00e9es', 6, 6, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'EDINF4GM', 'Project S4', 'Projet S4', 3, 6, 'Project', ''),
    # === B3 / L3 — Year 3 ===
    # Semester 7
    (BPROG, 'Bachelor', 'ELINF5AM', 'Systems Programming', 'Programmation syst\u00e8me', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5CM', 'Parallel Programming', 'Programmation parall\u00e8le', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5GM', 'Functional Programming and Introduction to Abstract Data Types', 'Programmation fonctionnelle et intro aux types abstraits', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5MM', 'Probability and Statistics: Data Analysis with Statistical Packages', 'Probabilit\u00e9s et Statistiques (Analyse des donn\u00e9es sous logiciels)', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5FM', 'Graph Theory', 'Graphes', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5BM', 'Theory of Automata and Formal Languages', 'Langages et automates', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5DM', 'Computer Networks 3', 'R\u00e9seaux 3', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5EM', 'Object-Oriented Programming 2', 'Programmation orient\u00e9e objet 2', 3, 7, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF5HM', 'Project S5', 'Projet S5', 3, 7, 'Project', ''),
    # Semester 8
    (BPROG, 'Bachelor', 'ELINF6AM', 'Computer Graphics, Image Processing and Analysis', 'Informatique graphique, traitement et analyse d\'image', 3, 8, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF6PM', 'Advanced Functional Programming and Abstract Data Types', 'Types abstraits et Programmation fonctionnelle avanc\u00e9e', 3, 8, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF6GM', 'Signal Processing', 'Traitement du signal', 3, 8, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF6DM', 'Software Engineering', 'G\u00e9nie logiciel', 3, 8, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF6CM', 'Computer Security (IT Security)', 'S\u00e9curit\u00e9 informatique', 3, 8, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF6BM', 'Databases 2', 'Base de donn\u00e9es 2', 3, 8, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF6RM', 'Artificial Intelligence', 'Intelligence artificielle', 3, 8, 'Obligatory', ''),
    (BPROG, 'Bachelor', 'ELINF6HM', 'Internship', 'Stage', 6, 8, 'Internship', '7-9 weeks'),
    # === M1 — Master Year 1 ===
    # Semester 1 (M1-S1)
    (MPROG, 'Master', '', 'Parallel Computing', 'Parall\u00e9lisme', 6, 1, 'Obligatory', ''),
    (MPROG, 'Master', '', 'Scientific Computing and Machine Learning', 'Calcul scientifique et apprentissage automatique', 6, 1, 'Obligatory', ''),
    (MPROG, 'Master', '', 'Advanced Algorithms', 'Algorithmique avanc\u00e9e', 6, 1, 'Obligatory', ''),
    (MPROG, 'Master', '', 'Theory of Languages', 'Th\u00e9orie des langages', 6, 1, 'Obligatory', ''),
    (MPROG, 'Master', '', 'Modeling, Design, and Collaborative Development', 'Mod\u00e9lisation, Conception, D\u00e9veloppement Collaboratif', 6, 1, 'Obligatory', ''),
    # Semester 2 (M1-S2)
    (MPROG, 'Master', '', 'Data Processing 1: Image, Sound and Text', 'Traitement de donn\u00e9es 1: image, son et texte', 6, 2, 'Obligatory', ''),
    (MPROG, 'Master', '', 'Artificial Intelligence 1', 'Intelligence artificielle 1', 6, 2, 'Obligatory', ''),
    (MPROG, 'Master', '', 'Computer Graphics 1', 'Informatique Graphique 1', 3, 2, 'Elective', '1 option au choix'),
    (MPROG, 'Master', '', 'Multi-Agent Systems', 'Syst\u00e8mes multi-agents', 3, 2, 'Elective', '1 option au choix'),
    (MPROG, 'Master', '', 'Research-Oriented Project', 'Travaux d\'initiation \u00e0 la Recherche-Projet', 6, 2, 'Obligatory', ''),
    (MPROG, 'Master', '', 'Internship / Stage', 'Stage', 6, 2, 'Elective', '1 option au choix; 7-9 weeks'),
    (MPROG, 'Master', '', 'Research and Development Project', 'Travaux d\'\u00c9tude et de Recherche', 6, 2, 'Elective', '1 option au choix'),
]

print(f'Total courses extracted: {len(raw_courses)}')
print(f'Bachelor: {sum(1 for r in raw_courses if r[1]=="Bachelor")}')
print(f'Master:   {sum(1 for r in raw_courses if r[1]=="Master")}')

In [ ]:
# Build DataFrame with course IDs
rows = []
for i, (prog, deg, code, name_en, name_orig, credits, sem, comp, notes) in enumerate(raw_courses, 1):
    row = make_row(prog, deg, code, name_en, name_orig, credits, sem, comp, notes)
    row['course_id'] = f'UFAR-{i:03d}'
    rows.append(row)

df = pd.DataFrame(rows)

# Reorder columns to match existing schema
col_order = ['course_id', 'university', 'program_name', 'program_code', 'degree_level',
             'course_code', 'course_name', 'course_name_original', 'credits', 'component',
             'semester', 'description', 'prerequisites', 'assessment', 'academic_year',
             'source_url', 'source_language', 'notes', 'course_name_en', 'description_en']
df = df[col_order]

print(f'DataFrame: {len(df)} rows, {len(df.columns)} columns')
print(f'\nBy program and degree:')
for (prog, deg), sub in df.groupby(['program_name', 'degree_level']):
    print(f'  {deg}: {prog} — {len(sub)} courses, {sub["credits"].sum()} ECTS')

print(f'\nBy semester:')
for (prog, sem), sub in df.groupby(['program_name', 'semester']):
    tag = 'B' if sub.iloc[0]['degree_level'] == 'Bachelor' else 'M'
    print(f'  {tag} Sem {sem}: {len(sub)} courses, {sub["credits"].sum()} ECTS')

In [ ]:
# Validate: check ECTS totals match the PDF header values
bachelor = df[df['degree_level'] == 'Bachelor']
master   = df[df['degree_level'] == 'Master']

# PDF states: BA = 69 ECTS, B1 = 60, B2 = 60, B3 = 60 (total ~249 for core IT courses)
# We exclude language/elective courses, so our totals will be lower
print('ECTS validation (IT courses only, excluding languages and general electives):')
for sem_range, label in [((1,2), 'BA/LA'), ((3,4), 'B1/L1'), ((5,6), 'B2/L2'), ((7,8), 'B3/L3')]:
    sub = bachelor[(bachelor['semester'] >= sem_range[0]) & (bachelor['semester'] <= sem_range[1])]
    print(f'  {label}: {sub["credits"].sum()} ECTS across {len(sub)} courses')

print(f'  M1:   {master["credits"].sum()} ECTS across {len(master)} courses')
print(f'\nTotal: {df["credits"].sum()} ECTS across {len(df)} courses')

In [ ]:
# Save
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT, index=False)
print(f'Saved to {OUTPUT}')
print(df[['course_id', 'course_name_en', 'credits', 'semester', 'degree_level', 'component']].to_string())